# Green & Fast Delivery — Machine Learning MVP

<br>

<p align="center">
  <img src="https://cdn-icons-png.flaticon.com/512/2972/2972185.png" width="120">
</p>

<br>


## Cadre académique

- **Module** : Machine Learning supervisé avec Python
- **Programme** : NEXA School IA — Master 1

<br>

## Équipe projet

- BAITE Camille
- CASTERAS Maxime
- DELGADO David
-
## Contexte

Ce projet s’inscrit dans le cadre du module de **Machine Learning supervisé avec Python**.

Il répond à un besoin métier formulé par Eco-Delivery :
optimiser la gestion des livraisons afin de réduire les délais et l’empreinte carbone.

L’objectif est de construire rapidement un **MVP fonctionnel** capable de fournir une première estimation prédictive à partir des données disponibles.

<br>



## Objectifs

- Mettre en place un pipeline de données complet (chargement, nettoyage, transformation)
- Gérer les valeurs manquantes et la qualité des données
- Identifier une variable cible pertinente
- Construire un modèle baseline simple
- Évaluer les performances du modèle
- Fournir une solution reproductible et exploitable

<br>

## Approche (XP / Agile)

Le projet suit une logique **Extreme Programming (XP)** :

- Livraison rapide d’un prototype fonctionnel
- Simplicité des solutions (baseline avant optimisation)
- Code lisible et modulaire
- Amélioration itérative

L’objectif n’est pas d’obtenir le meilleur modèle, mais de **livrer de la valeur rapidement**.

<br>

## Données

Dataset utilisé :
https://www.kaggle.com/datasets/gauravmalik26/food-delivery-dataset

Les données contiennent :
- Informations sur les commandes
- Données temporelles
- Informations sur les livraisons

<br>



## Pipeline

Le pipeline mis en place comprend :

1. Chargement des données
2. Nettoyage :
   - gestion des valeurs manquantes
3. Transformation :
   - variables numériques → normalisation
   - variables catégorielles → encodage
4. Modélisation :
   - modèle baseline (SGDClassifier)
5. Évaluation :
   - accuracy
   - classification report

<br>

## Modèle

Le modèle utilisé est :

- **SGDClassifier**

Choix volontaire :
- rapide
- simple
- adapté à une première itération

Ce modèle sert de **baseline** pour valider le pipeline.

<br>


<br>
<br>
<br>

## **1. Introduction du projet**

Ce projet vise à construire un modèle de Machine Learning permettant d’analyser et de prédire des comportements liés aux livraisons de repas à partir de données opérationnelles.

L’objectif est double :
- comprendre les facteurs influençant la performance des livraisons (temps, efficacité, répartition)
- développer un modèle baseline capable de produire une première prédiction exploitable

Le projet s’inscrit dans une logique MVP, avec une approche Agile (XP), privilégiant la livraison rapide d’un pipeline fonctionnel avant toute optimisation avancée.

In [5]:
import os
import pandas as pd
import numpy as np

from sklearn.pipeline import make_pipeline
from sklearn.compose import make_column_selector, make_column_transformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.model_selection import train_test_split
from sklearn.linear_model import SGDClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import classification_report, accuracy_score

import kagglehub

In [6]:
DATASET_NAME = "gauravmalik26/food-delivery-dataset"

dataset_path = kagglehub.dataset_download(DATASET_NAME)

print("Dataset path:", dataset_path)
print("Files:", os.listdir(dataset_path))

Dataset path: C:\Users\david\.cache\kagglehub\datasets\gauravmalik26\food-delivery-dataset\versions\1
Files: ['Sample_Submission.csv', 'test.csv', 'train.csv']


In [7]:
csv_files = [f for f in os.listdir(dataset_path) if f.endswith(".csv")]

dfs = {}

for file in csv_files:
    path = os.path.join(dataset_path, file)

    try:
        df = pd.read_csv(path)
    except:
        df = pd.read_csv(path, sep=";")

    dfs[file] = df
    print(f"{file} → shape: {df.shape}")

selected_file = max(dfs, key=lambda k: dfs[k].shape[1])

df = dfs[selected_file]

print("\nSelected file:", selected_file)
print(df.shape)
df.head()

Sample_Submission.csv → shape: (11399, 2)
test.csv → shape: (11399, 19)
train.csv → shape: (45593, 20)

Selected file: train.csv
(45593, 20)


,ID,Delivery_person_ID,Delivery_person_Age,Delivery_person_Ratings,Restaurant_latitude,Restaurant_longitude,Delivery_location_latitude,Delivery_location_longitude,Order_Date,Time_Orderd,Time_Order_picked,Weatherconditions,Road_traffic_density,Vehicle_condition,Type_of_order,Type_of_vehicle,multiple_deliveries,Festival,City,Time_taken(min)
0,0x4607,INDORES13DEL02,37,4.9,22.745049,75.892471,22.765049,75.912471,19-03-2022,11:30:00,11:45:00,conditions Sunny,High,2,Snack,motorcycle,0,No,Urban,(min) 24
1,0xb379,BANGRES18DEL02,34,4.5,12.913041,77.683237,13.043041,77.813237,25-03-2022,19:45:00,19:50:00,conditions Stormy,Jam,2,Snack,scooter,1,No,Metropolitian,(min) 33
2,0x5d6d,BANGRES19DEL01,23,4.4,12.914264,77.678400,12.924264,77.688400,19-03-2022,08:30:00,08:45:00,conditions Sandstorms,Low,0,Drinks,motorcycle,1,No,Urban,(min) 26
3,0x7a6a,COIMBRES13DEL02,38,4.7,11.003669,76.976494,11.053669,77.026494,05-04-2022,18:00:00,18:10:00,conditions Sunny,Medium,0,Buffet,motorcycle,1,No,Metropolitian,(min) 21
4,0x70a2,CHENRES12DEL01,32,4.6,12.972793,80.249982,13.012793,80.289982,26-03-2022,13:30:00,13:45:00,conditions Cloudy,High,1,Snack,scooter,1,No,Metropolitian,(min) 30
